<a href="https://colab.research.google.com/github/christpaul94/MD1912/blob/main/260809_twopparticle_scattering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qq "git+https://github.com/christpaul94/MD1912.git#subdirectory=MolecularDynamics"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.4/319.4 kB 27.7 MB/s eta 0:00:00


In [2]:
!pip install pykeops

In [ ]:
# --- Zelle 1: Imports & GPU Settings ---

# 1. Zuerst Torch importieren und konfigurieren
import torch
import math
import time

# Torch Settings für maximale Float32/TF32 Performance
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

import torch._dynamo
torch._dynamo.config.capture_scalar_outputs = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

# 2. Dann KeOps importieren und testen (weckt das config modul auf)
import pykeops
pykeops.test_torch_bindings() # Zwingt KeOps zur Initialisierung

# 3. Erst danach deine eigenen Module importieren
from TrappedAtomsSimulation.InitialisationV2 import initialize_one_temp_gaussian_state, calculate_dimensionless_scales
from TrappedAtomsSimulation.force_calculation import calculate_interaction_strength, pair_keops_fp, pair_force_keops_NxD, pair_potential_keops_NxD
from TrappedAtomsSimulation.integrators import run_verlet_simulation_HO, solve_harmonic_analytical, run_verlet_simulation_general, run_verlet_simulation_dynamic, run_harmonic_splitting_simulation, harmonic_fp, no_force_fp
from TrappedAtomsSimulation.plot_utils import plot_energy_and_error, plot_thermalization
from TrappedAtomsSimulation.trap_potential_extended import calculate_single_beam_X_AXIS, calculate_single_beam_Y_AXIS, calculate_crossed_beam_dipole_potential, calculate_U0, calculate_crossed_trap_frequencies

kB = 1.380649e-23  # J/K
c_light = 299_792_458.0 # m/s
pi = math.pi
m_rb87 = 86.909 * 1.66054e-27  # kg

print(f"Using device: {device}, Precision: {dtype}")

[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... 

In [ ]:
# --- Zelle 2: Physikalische Definitionen ---
# Atommasse 87Rb
m_rb87 = 86.909 * 1.66054e-27  # kg

# D1 Linie
omega_0_D1 = 2 * pi * 377.1074635e12 # rad/s
Gamma_D1 = 2 * pi * 5.746e6          # rad/s

# D2 Linie
omega_0_D2 = 2 * pi * 384.230484e12 # rad/s
Gamma_D2 = 2 * pi * 6.065e6         # rad/s

# Laserfrequenz
omega_L = 2 * pi * 280.179867e12    # rad/s

# Symmetrische Crossed Dipole Trap (gleiche Leistung & Waist für X und Y)
P_x = 1#5.18        # W
w0_x_SI = 41e-6   # m (41 µm)
zR_x_SI = 4.94e-3 # m (4.94 mm)

P_y = P_x
w0_y_SI = w0_x_SI
zR_y_SI = zR_x_SI

# Gemeinsame Atom/Laser-Parameter
atom_laser_params = {
    "omega_L": omega_L,
    "omega_0_D1": omega_0_D1, "Gamma_D1": Gamma_D1,
    "omega_0_D2": omega_0_D2, "Gamma_D2": Gamma_D2
}

# Berechne U0 (Potentialtiefe)
U0_x_SI = calculate_U0(P=P_x, w0=w0_x_SI, **atom_laser_params)
U0_y_SI = calculate_U0(P=P_y, w0=w0_y_SI, **atom_laser_params)

print(f"Potentialtiefe Strahl 1: {(U0_x_SI / kB) * 1e6:.1f} µK")
print(f"Potentialtiefe Strahl 2: {(U0_y_SI / kB) * 1e6:.1f} µK")

# Berechne effektive Fallenfrequenzen im Zentrum
trap_frequencies = calculate_crossed_trap_frequencies(
    U0_1=U0_x_SI, w0_1=w0_x_SI, zR_1=zR_x_SI,
    U0_2=U0_y_SI, w0_2=w0_y_SI, zR_2=zR_y_SI,
    m=m_rb87
)
TRAP_FREQUENCIES_HZ = (trap_frequencies['freq_x_hz'], trap_frequencies['freq_y_hz'], trap_frequencies['freq_z_hz'])

print(f"Zentrale Fallenfrequenzen: fx={TRAP_FREQUENCIES_HZ[0]:.0f} Hz, fy={TRAP_FREQUENCIES_HZ[1]:.0f} Hz, fz={TRAP_FREQUENCIES_HZ[2]:.0f} Hz")

In [ ]:
# --- Zelle 3: Simulations-Setup & Dimensionslose Skalierung ---
N_PARTICLES = 10000
TEMPERATURE_K = .5e-6  # 1 µK
SIMULATION_TIME_S = 0.05
TIMESTEP_S = 1e-3
SUBSTEPS = 1000

# Streulänge für Interaktion berechnen
r0_phys, C_phys = calculate_interaction_strength(15)

# 1. NEU: Skalen separat berechnen (Modulare Architektur)
scales = calculate_dimensionless_scales(
    temp_ref_k=TEMPERATURE_K,
    omega_phys_hz=TRAP_FREQUENCIES_HZ,
    mass_kg=m_rb87,
    precision=dtype,
    device=device
)

# 2. Thermischen Startzustand berechnen (mit übergebenen Skalen)
sim_init_params = initialize_one_temp_gaussian_state(
    n_particles=N_PARTICLES,
    temp_k=TEMPERATURE_K,
    scales=scales,
    t_end_s=SIMULATION_TIME_S,
    dt_s=TIMESTEP_S,
    r0_phys=r0_phys,
    C_phys=C_phys
)

# Skalen für die Falle entpacken
L0 = scales['L0_m']
E0 = scales['E0_J']
T0 = scales['T0_s']

# Parameter für das Dipolfallen-Potential-Modul packen
trap_params_extended = {
    "P_x": P_x, "P_y": P_y,
    "w0_x": w0_x_SI, "w0_y": w0_y_SI,
    "s0_x": zR_x_SI, "s0_y": zR_y_SI,
    "omega_L": omega_L,
    "omega_0_D1": omega_0_D1, "Gamma_D1": Gamma_D1,
    "omega_0_D2": omega_0_D2, "Gamma_D2": Gamma_D2,
    "L0": float(L0),
    "E0": float(E0)
}

# Basis Parameter für den Integrator packen
integrator_params = {
    "t_values": sim_init_params['t_values'],
    "q0": sim_init_params['q0'],
    "p0": sim_init_params['p0'],
    "precision_type": sim_init_params['precision_type'],
    "device": sim_init_params['device'],
    "substeps": SUBSTEPS,
}

In [ ]:
import time
import torch
from typing import Callable, Dict


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq

import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

def plot_energy_analysis(results, filename=None):
    t = results['times'].cpu().numpy()
    E_kin = results['kinetic_energy'].cpu().numpy()
    E_pot_pair = results['potential_energy_pair'].cpu().numpy()
    E_pot_trap = results['potential_energy_trap'].cpu().numpy()

    E_tot = E_kin + E_pot_pair + E_pot_trap
    E0 = E_tot[0]
    if abs(E0) < 1e-9:
        rel_dev = E_tot
        ylabel_dev = r"Absolute Deviation $\Delta E$"
    else:
        rel_dev = (E_tot - E0) / np.abs(E0)
        ylabel_dev = r"Relative Deviation $\Delta E / E_0$"

    # ---------------------------------------------------------
    # NEU: Maximalen Fehler suchen und ausgeben
    # ---------------------------------------------------------
    # Wir suchen den Index des größten *absoluten* Fehlers
    max_idx = np.argmax(np.abs(rel_dev))
    max_dev_val = rel_dev[max_idx]

    print("\n" + "="*50)
    print(f"MAXIMALER ENERGIEFEHLER: {max_dev_val:.4e}")
    print("="*50 + "\n")

    # ---------------------------------------------------------
    # Plot 1: Energie-Komponenten
    # ---------------------------------------------------------
    fig1, ax1 = plt.subplots(figsize=(4, 3))
    ax1.plot(t, E_kin, label=r'$E_{kin}$', color='tab:blue', linewidth=1.5)
    ax1.plot(t, E_pot_pair, label=r'$E_{int}$', color='tab:orange', linewidth=1.5)
    if np.any(np.abs(E_pot_trap) > 1e-9):
        ax1.plot(t, E_pot_trap, label=r'$E_{trap}$', color='tab:green', linewidth=1.5)
    ax1.plot(t, E_tot, label=r'$E_{tot}$', color='black', linestyle='--', linewidth=2.0, alpha=0.8)

    ax1.set_xlabel('Time [dimless]')
    ax1.set_ylabel('Energy [dimless]')
    ax1.legend()
    ax1.grid(True, linestyle=':', alpha=0.6)
    fig1.tight_layout()

    # ---------------------------------------------------------
    # Plot 2: Energie-Erhaltung (Abweichung)
    # ---------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(4, 3))
    ax2.plot(t, rel_dev, color='crimson', linewidth=1.5)

    # NEU: Dünne gestrichelte horizontale Linie beim Maximalwert
    ax2.axhline(y=max_dev_val, color='black', linestyle='--', linewidth=0.8, alpha=0.7)

    ax2.set_xlabel('Time [dimless]')
    ax2.set_ylabel(ylabel_dev)
    ax2.legend(loc='best', fontsize=8) # Legende hinzugefügt, damit man die Linie zuordnen kann
    ax2.grid(True, linestyle=':', alpha=0.6)
    ax2.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    fig2.tight_layout()
    print((f'Max Dev: {max_dev_val:.1e}'))
    # ---------------------------------------------------------
    # Speichern oder Anzeigen
    # ---------------------------------------------------------
    fig1.savefig(f"headson_energy.pdf", format='pdf', bbox_inches='tight')
    fig2.savefig(f"headson_deviation.pdf", format='pdf', bbox_inches='tight')

    plt.show() # Optional, damit du die Plots auch im Notebook/Editor siehst


# ==============================================================================
# ANALYTICAL SCATTERING THEORY
# ==============================================================================
def analytic_scattering(b_values_dense, V0, sigma, E_rel):
    def V(r): return V0 * np.exp(-r**2 / (2 * sigma**2))

    def denominator_sq(r, b, E):
        if r < 1e-6: return -1.0
        val = 1.0 - V(r)/E - (b**2)/(r**2)
        return val

    def integrand(r, b, E):
        denom = denominator_sq(r, b, E)
        if denom <= 0: return 0.0
        return b / (r**2 * np.sqrt(denom))

    theta_results = []
    for b in b_values_dense:
        if b < 1e-4:
            theta_results.append(180.0)
            continue
        try:
            r_min = brentq(denominator_sq, 0.001*sigma, max(b, 10.0)*sigma, args=(b, E_rel))
        except ValueError:
            r_min = b
        eps = 1e-5 * sigma
        integral, _ = quad(integrand, r_min + eps, np.inf, args=(b, E_rel), limit=100)
        theta_rad = np.pi - 2 * integral
        theta_results.append(np.degrees(theta_rad))
    return theta_results

In [ ]:
# Parameter
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

# Zeit-Array
t_start = 0.0
t_end = 12.5
dt_save = 0.05  # Wie oft gespeichert wird
t_values = torch.arange(t_start, t_end, dt_save, device=device, dtype=dtype)

# Startpositionen: Teilchen 1 links (-5), Teilchen 2 rechts (+5)
q0 = torch.tensor([
    [-3.5, 0.0, 0.0],
    [ 3.5, 0.0, 0.0]
], device=device, dtype=dtype)


# Startimpulse: Bewegen sich aufeinander zu
# Angenommen Masse m=1, Geschwindigkeit v=1
p0 = torch.tensor([
    [ 0.5, 0.0, 0.0],  # Nach rechts
    [-.5, 0.0, 0.0]   # Nach links
], device=device, dtype=dtype)

# Parameter für die Kräfte
trap_params = {} # Leer, da Null-Falle
sim_init_params['pair_force_params']['V0'] = 1
sim_init_params['pair_force_params']['sigma'] = 1
simulation_results = run_verlet_simulation_general(
    t_values=t_values,
    q0=q0,
    p0=p0,
    # Hier kommt die Dummy-Falle rein:
    trap_force_func=no_force_fp,
    trap_force_params=trap_params,
    # Hier deine Wechselwirkung:
    pair_force_func=pair_force_keops_NxD,
    pair_potential_func=pair_potential_keops_NxD,
    pair_force_params=sim_init_params['pair_force_params'],
    # Simulationseinstellungen
    precision_type=dtype,
    device=device,
    substeps=50, # Feine Schritte zwischen den Speicherpunkten für Genauigkeit
    silent=False
)

# Zugriff auf Ergebnisse
pos = plot_energy_analysis(simulation_results)
# Plotten oder Analysieren...

In [ ]:



# --- Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

t_values = torch.arange(0.0, 25, 0.01, device=device, dtype=dtype)
trap_params = {}

V0 = 1.0
sigma = 1.0
v_init = 0.5
E_rel = 0.25

b_values = np.arange(0.0, 5.5, 0.5).tolist()
results_list = []
scattering_angles = []

print(f"Starting simulation for {len(b_values)} impact parameters...")

for b in b_values:
    q0 = torch.tensor([[-5.0, b/2.0, 0.0], [5.0, -b/2.0, 0.0]], device=device, dtype=dtype)
    p0 = torch.tensor([[v_init, 0.0, 0.0], [-v_init, 0.0, 0.0]], device=device, dtype=dtype)

    res = run_verlet_simulation_general(
        t_values=t_values, q0=q0, p0=p0,
        trap_force_func=no_force_fp,
        trap_force_params=trap_params,
        pair_force_func=pair_force_keops_NxD,
        pair_potential_func=pair_potential_keops_NxD,
        pair_force_params={'V0': V0, 'sigma': sigma},
        precision_type=dtype, device=device,
        substeps=1, silent=True
    )

    pos = res['positions'].cpu()
    mom = res['momenta'].cpu()
    results_list.append((b, pos))

    p_in = mom[0, 0, :2]
    p_fin = mom[-1, 0, :2]
    cos_theta = torch.dot(p_in, p_fin) / (torch.norm(p_in) * torch.norm(p_fin))
    cos_theta = torch.clamp(cos_theta, -1.0, 1.0)
    theta_deg = torch.rad2deg(torch.acos(cos_theta)).item()
    scattering_angles.append(theta_deg)

# ==============================================================================
# --- Plotting ---
# ==============================================================================

# ---------------------------------------------------------
# Plot 1: Trajektorien (x-y Ebene)
# ---------------------------------------------------------
plt.figure(figsize=(5, 4))

first_m = 10
step_n  = 2

indices_to_plot = list(range(0, first_m, step_n))
colors = plt.cm.plasma(np.linspace(0, 1, len(indices_to_plot)))

indices_to_plot = indices_to_plot[:-1] # b=4 entfernen

for i in indices_to_plot:
    b, pos = results_list[i]
    color = colors[indices_to_plot.index(i)]
    # Geändert: b als Integer in der Legende ohne Nachkommastellen
    plt.plot(pos[:, 0, 0], pos[:, 0, 1], color=color, label=f'b = {int(b)}')
    plt.plot(pos[:, 1, 0], pos[:, 1, 1], color=color)

plt.xlabel("x [σ]")
plt.ylabel("y [σ]")
plt.legend(loc='best')
plt.axis('equal')
plt.xlim(-3.6, 3.6)
plt.ylim(-3.9, 3.9)
plt.grid(True, alpha=0.5)
plt.tight_layout()

plt.savefig("scattering_trajectories.pdf", format='pdf', bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# Plot 2: Streuwinkel Theorie vs MD
# ---------------------------------------------------------
plt.figure(figsize=(5, 4))

b_dense = np.linspace(0.01, 5, 200)
theta_theory = analytic_scattering(b_dense, V0, sigma, E_rel)
plt.plot(b_dense, theta_theory, 'k-', linewidth=1.5, label='Theory (Scattering Integral)')
plt.plot(b_values, scattering_angles, 'x', color='red', markersize=9, mew=2, label='MD Simulation')

plt.xlabel("Impact parameter b [σ]")
plt.ylabel(r"Scattering angle $\theta$ (°)")
plt.legend()
plt.grid(True)
plt.yticks(np.arange(0, 181, 30))
plt.tight_layout()

plt.savefig("scattering_angles_comparison.pdf", format='pdf', bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# Maximale Abweichung nur printen
# ---------------------------------------------------------
theta_theory_b = analytic_scattering(b_values, V0, sigma, E_rel)
deviations = np.abs(np.array(scattering_angles) - np.array(theta_theory_b))
max_idx = np.argmax(deviations)
max_dev_percent = (deviations[max_idx] / np.max(scattering_angles) * 100) if np.max(scattering_angles) > 0 else 0.0

print("\n" + "="*60)
print("MAXIMUM DEVIATION FROM THEORY")
print("="*60)
print(f"Maximum deviation : {max_dev_percent:.2f}%")
print(f"at b-value         : {b_values[max_idx]:.2f} σ")
print("="*60)

In [ ]:
# ==============================================================================
# SKIZZE FÜR b = 2.0 – MIT PARTIKELN, VEKTOREN, ASYMPTOTEN, STREUWINKEL
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Arc

# 1. Simulation für b = 2.0 (falls noch nicht in results_list enthalten)
b_target = 2.0
# Falls die Simulation für b=2.0 bereits in results_list steckt, kannst du sie extrahieren:
# pos_target = [pos for b, pos in results_list if abs(b - b_target) < 1e-6][0]
# Ansonsten führe sie separat aus:
if not any(abs(b - b_target) < 1e-6 for b, _ in results_list):
    q0 = torch.tensor([[-5.0, b_target/2.0, 0.0], [5.0, -b_target/2.0, 0.0]], device=device, dtype=dtype)
    p0 = torch.tensor([[v_init, 0.0, 0.0], [-v_init, 0.0, 0.0]], device=device, dtype=dtype)
    res_target = run_verlet_simulation_general(
        t_values=t_values, q0=q0, p0=p0,
        trap_force_func=no_force_fp,
        trap_force_params=trap_params,
        pair_force_func=pair_force_keops_NxD,
        pair_potential_func=pair_potential_keops_NxD,
        pair_force_params={'V0': V0, 'sigma': sigma},
        precision_type=dtype, device=device,
        substeps=1, silent=True
    )
    pos_target = res_target['positions'].cpu()
    mom_target = res_target['momenta'].cpu()
else:
    # Falls schon in results_list: finde den Eintrag
    pos_target = next(pos for b, pos in results_list if abs(b - b_target) < 1e-6)

# 2. Trajektorie für das obere Teilchen (Index 0) – das wird gestreut
x_upper = pos_target[:, 0, 0].numpy()
y_upper = pos_target[:, 0, 1].numpy()
# Unteres Teilchen (Index 1) – symmetrisch, aber wir zeichnen es nur als Punkt
x_lower = pos_target[:, 1, 0].numpy()
y_lower = pos_target[:, 1, 1].numpy()

# 3. Bestimme Ein‑ und Ausgangsrichtung (Anfang und Ende der Simulation)
n = len(x_upper)
n_fit = max(5, int(0.05 * n))

# Initiale Richtung (von links nach rechts)
x0_init = x_upper[:n_fit]
y0_init = y_upper[:n_fit]
slope_init, intercept_init = np.polyfit(x0_init, y0_init, 1)

# Finale Richtung (nach der Streuung)
x0_final = x_upper[-n_fit:]
y0_final = y_upper[-n_fit:]
slope_final, intercept_final = np.polyfit(x0_final, y0_final, 1)

# 4. Asymptoten als gestrichelte Linien zeichnen
x_asymp_init = np.array([x_upper[0] - 2.0, 4.0])
y_asymp_init = slope_init * (x_asymp_init - x_upper[0]) + y_upper[0]
x_asymp_final = np.array([-4.0, x_upper[-1] + 2.0])
y_asymp_final = slope_final * (x_asymp_final - x_upper[-1]) + y_upper[-1]

# 5. Streuwinkel θ aus der Änderung der Richtung berechnen
v_init_vec = np.array([1.0, slope_init])
v_init_vec = v_init_vec / np.linalg.norm(v_init_vec)
v_final_vec = np.array([1.0, slope_final])
v_final_vec = v_final_vec / np.linalg.norm(v_final_vec)
cos_theta = np.dot(v_init_vec, v_final_vec)
theta_deg = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
print(f"Streuwinkel (aus Simulation) = {theta_deg:.1f}°")


# ==============================================================================
# PLOT DER SKIZZE
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

# --- Trajektorie (durchgezogene Linie) ---
ax.plot(x_upper, y_upper, 'k-', linewidth=0.8, label='Trajektorie')
ax.plot(x_lower, y_lower, 'k-', linewidth=0.8)

# --- Asymptoten (gestrichelt) ---
ax.plot(x_asymp_init, y_asymp_init, 'gray', linestyle='dashed', linewidth=1, label=r'Asymptote (einlaufend)')
ax.plot(x_asymp_final, y_asymp_final, 'gray', linestyle='dotted', linewidth=1, label=r'Asymptote (auslaufend)')

# --- Partikel (deutlich größer: s=130) ---
idx_early = max(5, int(0.18 * n))
ax.scatter(x_upper[idx_early], y_upper[idx_early], color='black', s=130, zorder=5, label='Teilchen')
ax.scatter(x_lower[idx_early], y_lower[idx_early], color='black', s=130, zorder=5)

# --- Geschwindigkeitsvektoren (an den Anfangspositionen) ---
v0_norm = v_init_vec * 0.95
ax.arrow(x_upper[idx_early], y_upper[idx_early], v0_norm[0], v0_norm[1],
         head_width=0.12, head_length=0.2, fc='black', ec='black', length_includes_head=True, zorder=5)
# v1 auf 2 Uhr
ax.text(x_upper[idx_early] + 0.15, y_upper[idx_early] + 0.1, r'$\vec{v}_1$', ha='left', va='bottom', fontsize=11)

ax.arrow(x_lower[idx_early], y_lower[idx_early], -v0_norm[0], -v0_norm[1],
         head_width=0.12, head_length=0.2, fc='black', ec='black', length_includes_head=True, zorder=5)
# v2 auf 2 Uhr
ax.text(x_lower[idx_early] + 0.15, y_lower[idx_early] + 0.1, r'$\vec{v}_2$', ha='left', va='bottom', fontsize=11)

# --- Impact Parameter b ---
y_asymp_at_0 = slope_init * (0 - x_upper[0]) + y_upper[0]
b_val = b_target
y_other = -b_val/2

# Doppelpfeil für b
offset = 1.4
arrow = FancyArrowPatch((offset, y_other), (offset, y_asymp_at_0),
                        arrowstyle='<->', mutation_scale=15, linewidth=1,
                        color='black', clip_on=False)
ax.add_patch(arrow)
# b-Beschriftung
ax.text(offset + 0.15, 0.15, r'$b$', va='bottom', fontsize=12)


# --- Streuwinkel θ (Bogen zwischen Horizontale und Asymptote) ---
x_inter = (intercept_final - intercept_init) / (slope_init - slope_final)
y_inter = slope_init * x_inter + intercept_init

# Horizontale Hilfslinie (grau, gepunktet, unterhalb der Partikel mit zorder=1)
ax.plot([x_inter, x_inter + 2.5], [y_inter, y_inter], color='gray', linestyle='dotted', linewidth=1.0, zorder=1)

angle_final_rad = np.arctan2(v_final_vec[1], v_final_vec[0])
angle_final_deg = np.degrees(angle_final_rad)

# Bogen zeichnen
radius = 1.2
arc = Arc((x_inter, y_inter), 2*radius, 2*radius,
          theta1=0, theta2=angle_final_deg,
          color='black', linewidth=1)
ax.add_patch(arc)

# Beschriftung theta innerhalb des Bogens
mid_angle = angle_final_rad / 2
label_x = x_inter + radius * 0.65 * np.cos(mid_angle)
label_y = y_inter + radius * 0.65 * np.sin(mid_angle)
ax.text(label_x, label_y, r'$\theta$', fontsize=12, ha='center', va='center')


# --- Koordinatenachsen (sehr dünn: linewidth=0.4) ---
# --- Koordinatenachsen (sehr dünn: linewidth=0.4, aber größere Pfeile) ---
xlim = (-3.5, 3.5)
ylim = (-2, 3)
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# x-Achse
ax.arrow(xlim[0], 0, xlim[1]-xlim[0]-0.2, 0,
         head_width=0.15, head_length=0.25, fc='black', ec='black',
         length_includes_head=True, linewidth=0.4, zorder=0)
ax.text(xlim[1]-0.2, -0.15, r'$x$', fontsize=12)

# y-Achse
ax.arrow(0, ylim[0], 0, ylim[1]-ylim[0]-0.2,
         head_width=0.15, head_length=0.25, fc='black', ec='black',
         length_includes_head=True, linewidth=0.4, zorder=0)
ax.text(-0.2, ylim[1]-0.2, r'$y$', fontsize=12, va='center')

# --- Ästhetik ---
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("scattering_sketch.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# --- Zelle 5: Energie Plot ---
# Zeit in Sekunden umrechnen
t_tensor_phys = simulation_results['times'] * T0

# Hole die berechneten Energien aus dem Dictionary
# (Hinweis: Prüfe, unter welchem Key dein Integrator die Dipol-Energie speichert.
# Oft ist es noch als 'potential_energy_harmonic' gelabelt, auch wenn es der Dipol ist)
E_kin = simulation_results['kinetic_energy']
E_pot_trap = simulation_results.get('potential_energy_trap', simulation_results.get('potential_energy_harmonic'))
E_pot_pair = simulation_results['potential_energy_pair']

plot_energy_and_error(
    t_tensor_phys,
    E_kin,
    E_pot_trap,
    E_pot_pair
)

In [ ]:
# --- Zelle 6: Simulation Starten (Harmonische Falle) ---
print(f"\n>>> Starte N={N_PARTICLES} Simulation im idealen harmonischen Potential...")
start_time_harm = time.perf_counter()

# Wir nutzen die bereits fertig skalierte und umgerechnete Omega-Matrix
# aus unserer Initialisierungs-Zelle (inkl. 2*pi und dimensionsloser Skalierung)
trap_params_harmonic = {
    "omega_matrix": sim_init_params["omega_matrix"]
}

simulation_results_harm = run_verlet_simulation_general(
    trap_force_func=harmonic_fp,                 # <--- Ideales harmonisches Potential
    trap_force_params=trap_params_harmonic,      # <--- Passende Matrix übergeben
    pair_force_func=pair_force_keops_NxD,
    pair_potential_func=pair_potential_keops_NxD,
    pair_force_params=sim_init_params['pair_force_params'],
    **integrator_params                          # <--- Exakt selbe Startwerte (q0, p0, t)
)

duration_harm = time.perf_counter() - start_time_harm
print(f"Simulation erfolgreich abgeschlossen in {duration_harm:.2f} Sekunden.")

In [ ]:
# --- Zelle 7: Energie Plot (Harmonische Falle) ---

# Energien aus dem Dictionary der harmonischen Simulation auslesen
E_kin_harm = simulation_results_harm['kinetic_energy']
E_pot_trap_harm = simulation_results_harm.get('potential_energy_trap', simulation_results_harm.get('potential_energy_harmonic'))
E_pot_pair_harm = simulation_results_harm['potential_energy_pair']

print("--- Energiebilanz: Harmonisches Potential ---")
plot_energy_and_error(
    t_tensor_phys, # Die physikalische Zeitachse aus Zelle 5 können wir wiederverwenden
    E_kin_harm,
    E_pot_trap_harm,
    E_pot_pair_harm
)